# Dichroic bead control: S³M colour calls against filter-channel references

**Makes up Figure(s):** S78, `FigureSI_DichroicBeads.svg` (section 5), and the SI confusion table
`DichroicBeads_confusion_t0.csv`.

**Data:** BioStudies `S-BIAD3210`,
`2026_Multicolour_Paper/Ximea/Revision_2/20260929_dichroic_filtered_beads/ManualAcquisition/ND3p5/`:
36 fields of view (FOVs), ~19 MB each (~0.7 MB metadata), ~700 MB in all. Yellow-green (YG, FluoSpheres
F8803) and red (R, F8801) 100 nm beads on one coverslip, ND 3.5, Ximea camera. Each FOV is one `.ome.tif`
of 5 timepoints × 3 channels, interleaved (frame = 3 × timepoint + channel):

| channel | excitation | emission filters | sees |
|---|---|---|---|
| 0 | 488 nm | Di03-R488/561 + FF01-520/44 | YG beads (reference) |
| 1 | 561 nm | Di03-R488/561 + BLP01-568R + FF01-582/64 | R beads (reference) |
| 2 | 488 + 561 nm | Di03-R488/561 + NF03 quad notch | all beads: the S³M measurement |

Three power sets, 12 FOVs each: 488 nm at 2, 4 and 8 mW with 561 nm at 5, 10 and 20 %. The folder names
(`<set>_DC2_<n>`) are the only record of the powers.

**Figure S78:**
a)–c) filter transmission and bead emission in each channel;
d)–f) one FOV of the 2 mW / 5 % set in the YG, R and combined channels: persistent reference beads circled
in d) and e), S³M calls circled in green (YG) or red (R) in f);
g) the full-workflow confusion matrix for reference beads with **50–1,000 photoelectrons in their own filter
channel** (single-molecule-like signal), all three power sets, timepoint 0, with the missed beads by the
stage where they were lost;
h) photoelectrons of every scored reference bead in its own filter channel, stacked by power set, with the
ranges of g), i) and j) marked;
i), j) the same confusion for 1,000–2,000 and 2,000–5,000 photoelectrons.

Everything runs from the raw data: fit (section 1, skipped for FOVs that already have a fit unless
`REFIT`), filter and cluster, colour calls, reference tracking, figure. Signal is in detected
photoelectrons (`photoelectrons` = A_B/4 + A_G/2 + A_R/4). The analysis code is in `../_dichroic_beads.py`.

---
## What this analysis does, exactly (for the SI methods)

Pixel size 69 nm; distances in px are camera pixels. All thresholds are in `db.Settings` (section 0).

**Fitting (§1).** `AnalysisPipeline.fit` (mode `smlm`) on every frame of every FOV: spot detection on the raw
ADU frame with the calibrated per-pixel variance (false-alarm rate `pfa` = 1e-3, PSF wavelength 0.6 µm,
NA 1.49), then the STANDARD_DATA Gaussian fit, kept if it converged, lies inside its ROI and passes the
Δχ² fit gate (noise rate 0.01).

**Quality filter, clustering, colour cut (§2), identical in every channel.** A localisation is kept if it
has > 50 detected photoelectrons, both PSF widths between 75 and 250 nm, both width errors < 40 nm,
χ²_red < 3 and both localisation errors < 1 px, as in the Methods. There is no colour-error cut: cutting on
a localisation's own colour error would select on the quantity being tested. The kept localisations of each
FOV, all 15 frames together, are clustered with HDBSCAN (minimum cluster size 2); unclustered localisations
are dropped, then those with A_B ≥ 0.3.

**Colour calls (§2).** Per power set, a two-component Gaussian mixture on (A_R, A_G) of *every*
combined-channel localisation left, fitted without any reference information. The component with the
higher mean A_R is R. A localisation whose posterior is < 0.6 for both components is *unclassified*.

**Reference beads (§3): the ground truth, from the filter channels only.** In each FOV and reference channel
(0 → YG, 1 → R), spot detection (the same call and settings as the fit) on all 5 timepoints. Detections
within 2 px (138 nm) are one candidate, at its first detection. A candidate detected in ≥ 3 of the 5
timepoints is a *persistent reference bead*. A bead persistent in both reference channels (positions within
2 px) has no single true colour and is not scored (the count is printed in §3).

**Tracking and outcome (§3).** Each reference bead is followed, at each timepoint, in its own channel and in
the combined channel, through five stages, each a match within 2 px of its reference position (no channel
registration): detected → fitted → passed the quality filter → clustered → A_B < 0.3. In the combined
channel, a bead that passes every stage takes the colour call of the nearest S³M localisation within 2 px
(called YG / called R / unclassified); otherwise it is *missed*, labelled with the stage where it was lost.

**Which reference beads are scored.** Timepoint 0 only, so each bead is one independent event (95 % Wilson
score intervals). A bead is scored if, at timepoint 0, it also passes the full pipeline in its own filter
channel, with < 5,000 photoelectrons there: the same detection limit as S³M, decided in the reference
channel and never by the S³M outcome. (The Gaussian fitter is less reliable above 5,000 pe; single
molecules are below it.) Its photoelectron range in g), i), j) is the photoelectrons fitted in its own filter
channel, the only signal a missed bead has.

**The "no reference" row.** S³M calls at timepoint 0 (< 5,000 pe) with no persistent reference bead within
2 px, binned by their own (combined-channel) photoelectrons. They are not all false positives: the reference
set is incomplete (beads seen in < 3 timepoints of their own channel, or in neither filter channel).

---
## 0 — Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# _biostudies_download.py and _dichroic_beads.py live one level up (notebooks/figures/).
sys.path.insert(0, str(Path.cwd().resolve().parent))
import _biostudies_download as bd
import _dichroic_beads as db

import pyS3M.PlottingBase as PlottingBase
from pyS3M import SpectralFunctions
from pyS3M.AnalysisPipeline import AnalysisPipeline, FittingConfig
from pyS3M.Constants import AnalysisConfig

REFIT = False   # True: fit every FOV again, not only those without a fit

ST = db.Settings()                       # every threshold of the analysis (methods cell above)
SM_RANGE = (50, 1000)                    # g): single-molecule-like, own-channel photoelectrons
HIGH_RANGES = [(1000, 2000), (2000, 5000)]   # i), j)
EXAMPLE = ("488_2mW_561_5p", "DC2_6")    # d)-f): power set and FOV (as published)
FOVS = {s: [f"{s}_DC2_{n}" for n in range(1, 13)] for s in db.SETS}
REMOTE_ROOT = "2026_Multicolour_Paper/Ximea/Revision_2/20260929_dichroic_filtered_beads/ManualAcquisition/ND3p5"

# ── Paths ──────────────────────────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd().resolve().parents[2]
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = Path("outputs") / "DichroicBeads"   # git-ignored (bare outputs/ rule)
DATA_DIR  = OUT_ROOT / "data"                   # downloads; one folder per FOV
FIG_DIR   = OUT_ROOT / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

pipe = AnalysisPipeline(camera="ximea", config=AnalysisConfig(display=False))
pipe.load_calibration(CALIB_DIR)
fc = FittingConfig(peak_wavelength=0.6, pfa=1e-3, fit_gate="delta_chi2", noise_rate=0.01)
plotter = PlottingBase.PublicationPlotter()
print(f"Outputs    : {OUT_ROOT.resolve()}")

---
## 1 — Download and fit

Each FOV is downloaded to its own folder (skipped if already complete) and fitted unless it already has a
fit (`REFIT`). The first fit with a new ROI size / read noise calibrates the fit gate (~30 s, cached in
`Camera_Calibrations/fit_gate/`).

In [ ]:
fov_dirs = {}
for s, names in FOVS.items():
    fov_dirs[s] = []
    for name in names:
        files = bd.download_acquisition(f"{REMOTE_ROOT}/{name}", DATA_DIR / name)
        fov_dir = files["images"][0].parent
        if REFIT or not any(fov_dir.glob("*.h5")):
            pipe.fit(fov_dir, mode="smlm", fitting_config=fc)
        fov_dirs[s].append(fov_dir)
print({s: len(d) for s, d in fov_dirs.items()})

---
## 2 — Filter, cluster, colour calls and reference tracking

Per power set: quality cuts, HDBSCAN and the GMM colour calls (`s3m`, every combined-channel localisation);
then every reference bead followed through the pipeline (`track`; ~10 min of spot detection). Both are cached
in `OUT_ROOT` with everything they depend on (settings, FOVs, the fits' modification times) and reused while
that is unchanged.

In [ ]:
from dataclasses import asdict


def analyse():
    fov_data, s3m = {}, []
    for s in db.SETS:
        fov_data[s], sf = db.load_set(pipe, fov_dirs[s], ST)
        combined = sf[sf["frame"] % db.N_CHANNELS == 2]
        s3m.append(db.unmix(pipe, combined, ST).assign(set=s))
        print(f"{db.SET_LABELS[s]}: {sf['molecular_index'].nunique()} clusters, "
              f"{len(combined)} combined-channel localisations")
    track = pd.concat([db.track_fov(pipe, fc, ST, s, i, d, *fov_data[s][i])
                       for s in db.SETS for i, d in enumerate(fov_dirs[s])], ignore_index=True)
    return {"s3m": pd.concat(s3m, ignore_index=True), "track": track}


record = {"settings": asdict(ST), "fitting": fc.__dict__, "fovs": FOVS,
          "fits": {d.name: max(f.stat().st_mtime for f in d.glob("*.h5"))
                   for dirs in fov_dirs.values() for d in dirs}}
cache = db.cached(OUT_ROOT / "bead_analysis.h5", record, analyse)
s3m, track = cache["s3m"], cache["track"]

---
## 3 — S³M outcome of each reference bead

In [ ]:
scored, n_both = db.score(track, s3m, ST)

t0 = scored[scored["timepoint"] == 0]
events = t0[t0["own_kept"] & (t0["own_pe"] < ST.max_pe)]            # scored reference beads
noref = s3m[(s3m["frame"] == 2) & (s3m["reference"] == "no reference")
            & (s3m["photoelectrons"] < ST.max_pe)]                  # S3M calls with no reference
print(f"Timepoint 0: {len(events)} reference beads scored "
      f"({(events['bead_type'] == 'YG').sum()} YG, {(events['bead_type'] == 'R').sum()} R); "
      f"{len(noref)} S3M calls with no reference; {n_both} beads persistent in both channels not scored")

---
## 4 — Confusion table (SI)

Counts and 95 % Wilson intervals: all photoelectrons together, then each range of the figure.

In [ ]:
blocks = {"all (50–5,000 pe)": db.confusion_table(db.confusion_counts(events, noref)[0])}
for lo, hi in [SM_RANGE, *HIGH_RANGES]:
    ev = events[(events["own_pe"] >= lo) & (events["own_pe"] < hi)]
    nr = noref[(noref["photoelectrons"] >= lo) & (noref["photoelectrons"] < hi)]
    blocks[db.range_label(lo, hi)] = db.confusion_table(db.confusion_counts(ev, nr)[0])
table = pd.concat(blocks, names=["photoelectrons", "row"])
table.to_csv(OUT_ROOT / "DichroicBeads_confusion_t0.csv")
table

---
## 5 — Figure S78

Where each panel's data come from: a)–c) the filter and FluoSpheres spectra in `Spectra/spectral_data.duckdb`;
d)–f) timepoint 0 of FOV `EXAMPLE`; g)–j) the reference beads scored in §3 (timepoint 0, all three power
sets), split by the photoelectrons fitted in their own filter channel; the power sets present in each range
are written under its matrix.

In [ ]:
S_F = SpectralFunctions.Spectral_Funcs()
_, _, _, wavelength = S_F.getpixelefficiency()
spectra = db.channel_spectra(S_F, wavelength)

s, fov_name = EXAMPLE
fov = next(d for d in fov_dirs[s] if d.name.endswith(f"_{fov_name}"))
tif = next(fov.glob("*.ome.tif"))
frames = [pipe.sr.io.read_tiff(str(tif), frame=f) for f in range(db.N_CHANNELS)]   # timepoint 0
# Only what enters g)-j): this FOV's scored references and its no-reference S3M calls (by name, not index)
fov_events = events[events["fov_name"] == fov.name]
fov_noref = noref[noref["fov_name"] == fov.name]
print(f"{fov.name}: {(fov_events['bead_type'] == 'YG').sum()} YG and {(fov_events['bead_type'] == 'R').sum()} R "
      f"scored references, {len(fov_noref)} no-reference calls")

fig, counts = db.plot_figure(wavelength, spectra, frames, fov_events, fov_noref, 10_000 / ST.pixel_nm, events, noref,
                             ST, sm_range=SM_RANGE, high_ranges=HIGH_RANGES)
fig.savefig(FIG_DIR / "FigureSI_DichroicBeads.svg", dpi=600, format="svg")
plt.show()
for label, c in counts.items():
    print(label); print(c.to_string(), "\n")